## Setup

In [ ]:
!git clone --branch 4-pareto-test https://github.com/GoharKhachatryan/GenAI_project.git

In [ ]:
%cd GenAI_project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ln -s /content/drive/MyDrive/for_bostongene/data /content/GenAI_project/data
!ln -s /content/drive/MyDrive/for_bostongene/dataset.pt /content/GenAI_project/dataset.pt
!ln -s /content/drive/MyDrive/for_bostongene/pca_cache.npz /content/GenAI_project/pca_cache.npz
!ln -s /content/drive/MyDrive/for_bostongene/probe_weights.pt /content/GenAI_project/probe_weights.pt

## Info

This notebook performs the evaluation for the task 4.

## Imports

In [ ]:
import random
import time
import os

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from prepare_dataset import PairedCIFAR10

from pseudo_crossmodal import (
    extract_condition,
    load_pca,
)

from quality_probe import load_probe

from scripts.unet import UNet
from scripts.blocks import ScaleShiftResBlock
from scripts.model import ConditionalDenoiser
from scripts.diffusion import DDPM

from scripts.samplers import (
    ddpm_sample_respaced,
    ddim_sample,
)

## Reproducibility, Device and Configs

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

In [ ]:
CHECKPOINT_PATH = "/content/drive/MyDrive/for_bostongene/checkpoints/add_scale_shift/best_model.pt"

RESULTS_PATH = "/content/drive/MyDrive/for_bostongene/task4.pt"

## Test set & Results

In [ ]:
test_dataset = PairedCIFAR10("test")

eval_indices = list(
    range(len(test_dataset))
)

In [ ]:
index_generator = (
    torch.Generator()
    .manual_seed(SEED)
)

perm = torch.randperm(
    len(eval_indices),
    generator=index_generator,
).tolist()

eval_indices = [
    eval_indices[i]
    for i in perm
]

In [ ]:
eval_conditions = []
eval_labels = []

for idx in eval_indices:
    _, cond, label = test_dataset[idx]

    eval_conditions.append(cond)
    eval_labels.append(label)

eval_conditions = torch.stack(
    eval_conditions
)

eval_labels = torch.tensor(
    eval_labels
)

print(eval_conditions.shape)
print(eval_labels.shape)

In [ ]:
def save_results(results, path=RESULTS_PATH):
    torch.save(results, path)

In [ ]:
if os.path.exists(RESULTS_PATH):
    results = torch.load(
        RESULTS_PATH,
        map_location="cpu",
    )
    print(f"Loaded {len(results)} existing results.")
else:
    results = {}

## Model & Weights

In [ ]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
)

In [ ]:
cond_mean = checkpoint["cond_mean"]
cond_std = checkpoint["cond_std"]

eval_conditions_norm = (
    eval_conditions - cond_mean
) / (cond_std + 1e-8)

In [ ]:
unet = UNet(
    in_channels=3,
    out_channels=3,
    base_channels=64,
    emb_dim=256,
    block_cls=ScaleShiftResBlock,
)

model = ConditionalDenoiser(
    unet=unet,
    time_dim=256,
    cond_dim=16,
    emb_dim=256,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

## Sample fixed x_init

In [ ]:
noise_generator = (
    torch.Generator()
    .manual_seed(SEED)
)

x_init_all = torch.randn(
    len(eval_indices),
    3,
    32,
    32,
    generator=noise_generator,
)

print(x_init_all.shape)

## Diffusion, PCA, Classifier & Cycle

In [ ]:
diffusion = DDPM(
    timesteps=1000,
    schedule="cosine",
    device=device,
)

pca_state = load_pca()

In [ ]:
probe = load_probe(
    device=str(device)
)

probe.eval()

In [ ]:
def compute_cycle_metrics(
    cond_true,
    cond_hat,
    cond_std,
):
    error = cond_true - cond_hat

    raw_l2 = torch.linalg.vector_norm(
        error,
        dim=1,
    )

    standardized_error = (
        error
        / (cond_std + 1e-8)
    )

    standardized_l2 = torch.linalg.vector_norm(
        standardized_error,
        dim=1,
    )

    feature_mae = (
        error.abs().mean(dim=0)
    )

    feature_mae_std = (
        standardized_error.abs().mean(dim=0)
    )

    return {
        "raw_l2": raw_l2,
        "standardized_l2": standardized_l2,
        "feature_mae": feature_mae,
        "feature_mae_std": feature_mae_std,
    }

## Experiment configs

In [ ]:
STEP_COUNTS = [
    500,
    100,
    50,
    20,
    10,
]

SAMPLERS = [
    "ddpm",
    "ddim",
]

BATCH_SIZE = 128
DDIM_ETA = 0.0

## Evaluating function (for both metrics at once)

In [ ]:
@torch.no_grad()
def evaluate_sampler(
    model,
    probe,
    sampler_name,
    num_steps,
    cond_original,
    cond_normalized,
    labels,
    x_init_all,
):
    all_cond_hat = []

    correct = 0
    total = 0

    start_time = time.time()

    for start in range(0, len(cond_original), batch_size):
        end = min(start + batch_size, len(cond_original))

        cond_batch = cond_normalized[start:end].to(device)
        x_init_batch = x_init_all[start:end].to(device)
        label_batch = labels[start:end].to(device)

        # Generate
        if sampler_name == "ddpm":
            generated = ddpm_sample_respaced(
                model=model,
                ddpm=diffusion,
                cond=cond_batch,
                num_steps=num_steps,
                x_init=x_init_batch,
            )

        elif sampler_name == "ddim":
            generated = ddim_sample(
                model=model,
                ddpm=diffusion,
                cond=cond_batch,
                num_steps=num_steps,
                eta=0.0,
                x_init=x_init_batch,
            )

        # Cycle metric
        cond_hat = extract_condition(
            generated,
            pca_state,
        )

        all_cond_hat.append(cond_hat.cpu())

        # External quality probe
        logits = probe(generated)

        predictions = logits.argmax(dim=1)

        correct += (
            predictions == label_batch
        ).sum().item()

        total += label_batch.numel()

    cond_hat = torch.cat(all_cond_hat, dim=0)

    cycle_metrics = compute_cycle_metrics(
        cond_true=cond_original,
        cond_hat=cond_hat,
        cond_std=cond_std,
    )

    return {
        **cycle_metrics,
        "conditional_accuracy": correct / total,
        "runtime_sec": time.time() - start_time,
    }

## Evaluation loop

In [ ]:
for sampler_name in SAMPLERS:
    for num_steps in STEP_COUNTS:

        key = (
            sampler_name,
            num_steps,
        )

        torch.manual_seed(SEED)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)

        result = evaluate_sampler(
            model=model,
            probe=probe,
            sampler_name=sampler_name,
            num_steps=num_steps,
            cond_original=eval_conditions,
            cond_normalized=eval_conditions_norm,
            labels=eval_labels,
            x_init_all=x_init_all,
            cond_std=cond_std,
            pca_state=pca_state,
            diffusion=diffusion,
            batch_size=BATCH_SIZE,
            eta=0.0,
        )

        results[key] = result

        save_results(results, path=RESULTS_PATH)

## Results and Plots

In [ ]:
rows = []

for (sampler, steps), result in results.items():

    rows.append({
        "sampler": sampler.upper(),
        "steps": steps,

        "cycle_l2":
            result["raw_l2"].mean().item(),

        "standardized_cycle_l2":
            result["standardized_l2"].mean().item(),

        "conditional_accuracy":
            result["conditional_accuracy"],

        "runtime_sec":
            result["runtime_sec"],
    })

summary_df = pd.DataFrame(rows)

summary_df = summary_df.sort_values(
    ["sampler", "steps"],
    ascending=[True, False],
)

summary_df

In [ ]:
# A plot per metric
for sampler in ["DDPM", "DDIM"]:
    df = summary_df[
        summary_df["sampler"] == sampler
    ].sort_values("steps")

    plt.plot(
        df["steps"],
        df["standardized_cycle_l2"],
        marker="o",
        label=sampler,
    )

plt.xlabel("Sampling steps")
plt.ylabel("Standardized cycle L2")
plt.title("Cycle consistency vs sampling budget")
plt.legend()
plt.show()

In [ ]:
for sampler in ["DDPM", "DDIM"]:
    df = summary_df[
        summary_df["sampler"] == sampler
    ].sort_values("steps")

    plt.plot(
        df["steps"],
        df["conditional_accuracy"],
        marker="o",
        label=sampler,
    )

plt.xlabel("Sampling steps")
plt.ylabel("Conditional accuracy")
plt.title("Conditional accuracy vs sampling budget")
plt.legend()
plt.show()

In [ ]:
# the actual Pareto plot
for sampler in ["DDPM", "DDIM"]:
    df = summary_df[
        summary_df["sampler"] == sampler
    ]

    plt.scatter(
        df["standardized_cycle_l2"],
        df["conditional_accuracy"],
        label=sampler,
    )

    for _, row in df.iterrows():
        plt.annotate(
            f'{int(row["steps"])}',
            (
                row["standardized_cycle_l2"],
                row["conditional_accuracy"],
            ),
        )

plt.xlabel(
    "Standardized cycle L2 ↓"
)

plt.ylabel(
    "Conditional accuracy ↑"
)

plt.title(
    "Sampler quality trade-off"
)

plt.legend()
plt.show()